# Training Check in Colab

This notebook is made for a simple Colab training run.

It does four things:
- moves into the project folder in the Colab runtime
- installs the project dependencies
- loads the dataset and trains the model
- evaluates and saves the trained model


In [ ]:
import sys
import tensorflow as tf

# Quick check that Colab sees TensorFlow and the GPU.
print(sys.executable)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


In [ ]:
import os
from pathlib import Path

from google.colab import files

# Upload the 7z archive from your computer.
# The archive should contain the DAT255_Bayesian project folder.
uploaded = files.upload()
archive_name = next(iter(uploaded))
archive_path = Path("/content") / archive_name

# Install 7z support in Colab and extract the project into /content.
!apt-get -qq install -y p7zip-full
!7z x -y "{archive_path}" -o/content

# Some archives contain the project folder, others contain the files directly.
if Path("/content/DAT255_Bayesian/bayesian_cv").exists():
    REPO_DIR = Path("/content/DAT255_Bayesian")
else:
    REPO_DIR = Path("/content")

os.chdir(REPO_DIR)
print(Path.cwd())


In [ ]:
# Install the same dependencies as the project uses locally.
%pip install -r requirements.txt


In [ ]:
from dataclasses import replace

import tensorflow as tf

from bayesian_cv.config import get_config
from bayesian_cv.data import load_datasets
from bayesian_cv.model import build_model

# Start from the shared config and only override what we want for this run.
config = replace(
    get_config(),
    epochs=10,
)

print(config)


In [ ]:
# The data loader downloads ImageNette automatically the first time.
train_ds, val_ds, test_ds = load_datasets(config)

print("Datasets loaded successfully.")


In [ ]:
# Build the CNN from the project code.
model = build_model(config)

print("Model built successfully.")


In [ ]:
# Create the save folder and keep the best model during training.
config.model_path.parent.mkdir(parents=True, exist_ok=True)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=config.model_path,
        monitor="val_loss",
        save_best_only=True,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config.epochs,
    callbacks=callbacks,
)

print(f"Epochs completed: {len(history.history.get('loss', []))}")


In [ ]:
# Evaluate on the held-out split after training.
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")


In [ ]:
# Save the final model so it can be reused later.
model.save(config.model_path)

print(f"Saved model to: {config.model_path}")
